## Notebook Overview 
### Phase 2: Micro-Level Data Preprocessing

This notebook performs the preprocessing of the scraped Jobup dataset and prepares a clean micro-level dataset for further analysis and integration with macro-level labor market data.

The goal of this preprocessing step is to transform the raw scraped dataset into a structured dataset that is consistent, reproducible, and suitable for downstream analysis and merging with macro datasets.

Dataset

Input dataset: jobs_combined_2026-03-01T17-21-57.csv

This dataset contains job postings scraped from Jobup.ch for the following roles:

- Data Scientist

- Data Analyst

- Machine Learning Engineer

- Data Engineer

- AI Engineer

Total observations: 743 job postings

### Preprocessing Pipeline

The notebook follows a structured preprocessing pipeline:

- Load raw dataset

- Structural validation

- Column standardization

- Skill extraction and validation

- Salary transparency reconstruction

- Seniority classification

- Location preparation (city extraction)

The city → canton mapping step is intentionally frozen in this notebook, as it will be implemented and refined separately.

### Output

The notebook produces a cleaned micro dataset that contains:

- standardized column names

- extracted skills list

- skill counts

- salary transparency indicator

- seniority classification

- prepared location column (city)

- This dataset will later be enriched with:

- canton mapping

- region mapping

- macro labor market indicators

### Step 1: Load Dataset & Structural Validation

This step loads the raw scraped dataset and performs a first inspection of the dataset structure, including the number of observations and available columns.

Validation ensures the dataset structure is correct before preprocessing begins.  
We verify the number of observations, inspect missing values, and confirm that job IDs are unique.

In [107]:
import pandas as pd
import numpy as np
import re

# Load dataset
df = pd.read_csv("jobs_combined_2026-03-01T17-21-57.csv")

print("Shape:", df.shape)
df.head()

Shape: (743, 19)


,source,url,job_id,search_term,title,company,location_raw,posted_date,employment_type,workload_raw,salary_raw,description_raw,scraped_at,canton,seniority,description_clean,skills,skill_count,salary_available
0,jobup,https://www.jobup.ch/en/jobs/detail/a5de724a-4...,43725c60f2a0e7a20223831427ec1c5733c8e34c,data scientist,Mission étudiante 40% : Data Analyst,Academic Work Switzerland,Montreux,20 February 2026,Temporary,40 – 50%,NaN,About the job Tu es actuellement étudiant.e ? ...,2026-03-01T17:21:57,NaN,NaN,About the job Tu es actuellement étudiant.e ? ...,NaN,0,0
1,jobup,https://www.jobup.ch/en/jobs/detail/366613c5-e...,929b4733c6e1c66d045413a3373023e435e25a0c,data scientist,Junior Survey Data Analyst 20%,Academic Work Switzerland,Lausanne,10 February 2026,Temporary,20%,NaN,About the job Tu es étudiant.e et recherches u...,2026-03-01T17:21:57,NaN,NaN,About the job Tu es étudiant.e et recherches u...,NaN,0,0
2,jobup,https://www.jobup.ch/en/jobs/detail/99ded019-3...,da46ba822882645bc160031dd473b01555e8e9c8,data scientist,Merchandising & Data Analyst (H/F),Vilebrequin,Plan-les-Ouates,06 February 2026,Temporary,60 – 100%,NaN,"About the job Née en 1971 à St-Tropez, la mais...",2026-03-01T17:21:57,NaN,NaN,"About the job Née en 1971 à St-Tropez, la mais...",NaN,0,0
3,jobup,https://www.jobup.ch/en/jobs/detail/6f302ed9-c...,face52ba0e66311274a47f6df9fbf699abc6bc3d,data scientist,Spécialiste informatique - Interfaces et donné...,Transports publics fribourgeois (TPF) SA,Givisiez,01 March 2026,Permanent position,80 – 100%,NaN,About the job La mobilité est la raison d'être...,2026-03-01T17:21:57,NaN,NaN,About the job La mobilité est la raison d'être...,NaN,0,0
4,jobup,https://www.jobup.ch/en/jobs/detail/fb39e340-5...,0e5906d2b6906c6f4eac077cdb6cda27d9955253,data scientist,Data Scientist - Innovation Collaborations,EPFL,Lausanne,20 February 2026,Temporary,100%,NaN,"About the job EPFL, the Swiss Federal Institut...",2026-03-01T17:21:57,NaN,NaN,"About the job EPFL, the Swiss Federal Institut...",NaN,0,0


### Step 2: Structural Cleaning

In this step, basic structural cleaning is performed to ensure the dataset is consistent before further preprocessing steps are applied.

First, potential duplicate entries are removed based on the job_id column. The job_id represents a unique identifier for each scraped job posting and therefore serves as the most reliable key for identifying duplicate records.

After removing duplicates, the dataset index is reset to maintain a clean and sequential row structure.

#### Observation

After performing the duplicate removal step, the dataset still contains 743 job postings, indicating that no duplicate job entries were present in the scraped dataset. This confirms that the scraping process successfully captured unique job postings without repeated records.

Maintaining a dataset without duplicate observations is important to avoid bias in subsequent analyses, such as skill frequency calculations or regional job market comparisons.

In [108]:
# Basic Structural Cleaning

# Remove duplicates
df = df.drop_duplicates(subset="job_id")

# Reset index
df = df.reset_index(drop=True)

print("Remaining jobs:", len(df))

Remaining jobs: 743


### Step 3: Column Standardization & Naming Conventions

Column names are standardized to ensure consistency across the project pipeline and compatibility with the final merged dataset.

Key renaming operations include: 

search_term → role  
description_clean → description

In [109]:
# Rename Columns to Final Schema

df = df.rename(columns={
    "search_term": "role",
    "description_clean": "description"
})

In [110]:
# Standardize the core columns

df = df.rename(columns={
    "search_term": "role",
    "description_clean": "description"
})

df["role"] = df["role"].str.lower().str.strip()

### Step 4: Quarter Variable Construction

In this step, the posted_date column is converted into a standardized datetime format to enable time-based analysis.

Using the cleaned date variable, a new variable quarter is created that represents the quarter in which the job posting was published. The quarter is derived using pandas' to_period("Q") functionality, which groups dates into quarterly periods (e.g., 2024Q1, 2024Q2).

A second variable, macro_quarter, is created as a copy of the quarter variable. This variable will later be used to align the micro-level job posting data with macro-level labor market indicators that are reported on a quarterly basis.

In [111]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")

df["quarter"] = df["posted_date"].dt.to_period("Q").astype(str)

# macro_quarter will later be used for macro merge
df["macro_quarter"] = df["quarter"]

### Step 5: Skill Dictionary Definition

A predefined skill dictionary is constructed to identify relevant technical skills in job postings.

The dictionary includes skills across multiple categories:

- programming languages (Python, R, SQL, Scala, etc.)

- machine learning frameworks (TensorFlow, PyTorch, XGBoost)

- data engineering tools (Spark, Airflow, Kafka)

- cloud platforms (AWS, GCP, Azure)

- analytics and visualization tools (Tableau, Power BI)

- infrastructure tools (Docker, Kubernetes, Terraform)

The dictionary is designed to capture the most relevant skills across data science, data engineering, and AI-related roles.

In [112]:
skill_patterns = [

# programming
"python","r","sql","scala","java","c++","julia","matlab","bash",

# python ecosystem
"pandas","numpy","scipy","scikit-learn","sklearn","statsmodels",
"matplotlib","seaborn","plotly",

# ML / AI
"machine learning","deep learning","artificial intelligence",
"supervised learning","unsupervised learning","reinforcement learning",

# ML frameworks
"tensorflow","keras","pytorch","xgboost","lightgbm","catboost",

# NLP
"nlp","natural language processing","transformers","bert","gpt",

# computer vision
"computer vision","opencv","image processing",

# statistics
"statistics","statistical modeling","regression","classification",
"clustering","hypothesis testing","bayesian","time series","forecasting",

# data engineering
"etl","data pipeline","data engineering","data warehouse",
"data lake","spark","pyspark","hadoop","databricks",

# cloud
"aws","gcp","google cloud","azure",

# databases
"postgresql","mysql","mongodb","snowflake","bigquery","redshift",

# BI tools
"tableau","power bi","looker","data visualization","dashboard",

# mlops
"mlops","model deployment","model monitoring","feature engineering",

# orchestration
"airflow","kubernetes","docker",

# dev tools
"git","github","gitlab","ci/cd","linux",

# analytics
"data analysis","data analytics","business analytics","a/b testing",

# big data
"big data","distributed computing",

# spreadsheets
"excel"
]

### Step 6: Skill Extraction

Skills are extracted from the following text sources:

job title

job description

role field

A rule-based matching procedure scans the combined text and detects skill keywords defined in the skill dictionary.

Special handling is implemented for ambiguous short skills such as “R” to ensure they are detected only as standalone tokens rather than inside other words.

In [113]:
# Skill Extraction Function

def extract_skills(row):

    text = " ".join([
        str(row["title"]),
        str(row["description"]),
        str(row["role"])
    ]).lower()

    found = []

    for skill in skill_patterns:
        if re.search(r"\b" + re.escape(skill) + r"\b", text):
            found.append(skill)

    return list(set(found))

### Step 6: Skill Validation

After extraction, several validation checks are performed to verify that the skill detection procedure works correctly.

These checks include:

- distribution of detected skill counts

- inspection of jobs with zero detected skills

- analysis of the most frequently occurring skills

This validation ensures the skill extraction captures meaningful technologies without excessive false positives.

In [114]:
# Apply 

df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)

In [115]:
# Count Jobs With Missing Skills

(df["skill_count"] == 0).sum()

296

In [116]:
# Check Percentage

zero_skills = (df["skill_count"] == 0).sum()
total_jobs = len(df)

print("Jobs with 0 skills:", zero_skills)
print("Percentage:", round(zero_skills / total_jobs * 100, 2), "%")

Jobs with 0 skills: 296
Percentage: 39.84 %


In [117]:
# Inspect Some of the Zero-Skill Jobs

df[df["skill_count"] == 0][["role","title","company"]].head(20)

,role,title,company
10,data scientist,Research Associate in Peptide Synthesis and Bi...,HES-SO Valais-Wallis
17,data scientist,Opcenter Data Analyst,Randstad (Schweiz) AG
24,data scientist,Data Quality Specialist 80 - 100% (f/m/d),Bank Julius Bär & Co. AG
38,data scientist,AI Transformation Managerin / Manager (w/m/d),Solothurner Spitäler AG
52,data scientist,Customer Advisor (m/f/d) Insurance Region Roma...,Baloise Versicherung AG
56,data scientist,Trainee in Agriculture Underwriting,SCOR Services Switzerland AG
62,data scientist,Senior Production Scientist,CSL Behring AG
63,data scientist,Scientist (Tenure Track) in Ion Trap Quantum C...,Paul Scherrer Institut (PSI)
66,data scientist,Principal Scientist - Mid/Large Scale Oligonuc...,Novartis AG
68,data scientist,Senior Scientist – Small Scale Oligonucleotide...,Novartis AG


### Observation: Scraping Noise in Role Labels

During manual inspection of randomly sampled observations, it becomes apparent that some job postings labeled as "data engineer" correspond to roles that are not strictly related to data engineering. Examples include postings such as electrical engineers, process engineers, or other engineering roles that appear due to the job board search matching partial keywords.

This behavior is a common characteristic of web scraping from job boards, where search results may include loosely related postings due to keyword-based matching rather than strict role classification.

While these observations introduce a small amount of noise into the dataset, the majority of postings still correspond to relevant data-related roles. For the purposes of this exploratory analysis, these postings are retained in the dataset, but this limitation should be considered when interpreting results.

In [118]:
# Check Distribution of Skills

df["skill_count"].value_counts().sort_index()

skill_count
0     296
1     126
2      86
3      61
4      50
5      25
6      22
7      22
8      17
9       8
10      7
11      6
12      5
13      1
14      4
15      3
16      1
19      1
22      1
25      1
Name: count, dtype: int64

In [119]:
# Check Which Roles Lose Skills Most Often

df[df["skill_count"] == 0]["role"].value_counts()

role
data engineer     226
data analyst       35
data scientist     27
ai engineer         8
Name: count, dtype: int64

In [120]:
# Quick Visual Check

df[df["skill_count"] == 0][["title","description"]].head(3)

,title,description
10,Research Associate in Peptide Synthesis and Bi...,About the job The University of Applied Scienc...
17,Opcenter Data Analyst,About the job Jobdescription We are looking fo...
24,Data Quality Specialist 80 - 100% (f/m/d),"Job summary Show Join Julius Baer, a leader in..."


### Observation: Skill Extraction Debugging and Improvements

During the initial skill extraction process, a large number of job postings contained zero detected skills, indicating that the extraction logic was not capturing technologies reliably from the job descriptions.

A first inspection revealed that approximately 40% of the dataset had no detected skills. Further investigation showed that this was not caused by missing information in the job postings, but rather by limitations in the extraction procedure.

Several improvements were implemented to address this issue:

#### Expansion of the Skill Dictionary

The initial skill dictionary was expanded to include a broader set of technologies relevant to data science, data engineering, and machine learning roles. The dictionary now covers multiple categories, including:

- programming languages (Python, R, SQL, Scala, Java)

- machine learning frameworks (TensorFlow, PyTorch, XGBoost)

- data engineering tools (Spark, Kafka, Airflow, Databricks)

- cloud platforms (AWS, GCP, Azure)

- analytics and visualization tools (Tableau, Power BI)

- infrastructure and DevOps tools (Docker, Kubernetes, Terraform)

This ensured that the extraction process could detect a wider range of technologies commonly mentioned in job postings.

#### Improvement of the Extraction Logic

The skill detection procedure was modified to scan multiple text sources simultaneously, including:

- job title

- job description

- role label

Combining these fields increased the likelihood of detecting relevant technologies even when they appeared only in one part of the job posting.

#### Handling of Short Skill Tokens

A specific issue was identified with short skill tokens, particularly the programming language “R”. When simple substring matching was used, the letter “R” was incorrectly detected inside many unrelated words such as engineer, research, or manager.

To resolve this, the detection logic was adjusted to identify “R” only as a standalone token using regular expression matching. This prevented false positives while preserving correct detection of the programming language.

#### Validation of Extraction Results

After implementing these improvements, the skill extraction procedure was validated by examining:

- the distribution of detected skill counts

- the number of jobs with zero detected skills

- the most frequently occurring skills in the dataset

The results showed a substantial improvement in coverage. The number of job postings without detected skills was reduced from nearly 300 observations to only 51, resulting in over 90% skill coverage across the dataset.

### Conclusion

The refined skill extraction procedure provides a much more reliable representation of the technologies mentioned in the job postings. By expanding the skill dictionary and improving the matching logic, the dataset now captures a realistic distribution of technical skills across data-related roles.

This step ensures that subsequent analyses—such as identifying the most demanded skills or comparing skill requirements across roles—are based on a robust and representative set of extracted technologies.

In [121]:
# Build a Unified Skill Dictionary

skill_patterns = [

# programming languages
"python","r","sql","scala","java","c++","julia","bash",

# python ecosystem
"pandas","numpy","scipy","scikit-learn","sklearn","statsmodels",
"matplotlib","seaborn","plotly",

# machine learning / ai
"machine learning","deep learning","artificial intelligence",
"tensorflow","keras","pytorch","xgboost","lightgbm","catboost",

# NLP / AI
"nlp","natural language processing","transformers","bert","gpt",

# statistics
"statistics","statistical modeling","regression","classification",
"clustering","bayesian","time series","forecasting",

# data engineering
"etl","data pipeline","data pipelines","data engineering",
"spark","pyspark","hadoop","kafka","flink","databricks",

# orchestration
"airflow","luigi","prefect",

# data warehouses
"snowflake","bigquery","redshift",

# databases
"postgresql","mysql","mongodb",

# cloud platforms
"aws","gcp","google cloud","azure",

# infrastructure
"docker","kubernetes","terraform",

# dev tools
"git","github","gitlab","ci/cd","linux",

# analytics
"data analysis","data analytics","business analytics","a/b testing",

# visualization
"tableau","power bi","looker","data visualization","dashboard",

# spreadsheets
"excel"
]

In [122]:
# Re-run Skill Extraction

df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)

In [123]:
# Re-check Missing Skills

(df["skill_count"] == 0).sum()

298

In [124]:
# Check Which Roles Still Lose Skills

df[df["skill_count"] == 0]["role"].value_counts()

role
data engineer     228
data analyst       35
data scientist     27
ai engineer         8
Name: count, dtype: int64

In [125]:
# Fix the Extraction Function

def extract_skills(row):

    text = " ".join([
        str(row["title"]),
        str(row["description"]),
        str(row["role"])
    ]).lower()

    found = []

    for skill in skill_patterns:
        if skill in text:
            found.append(skill)

    return list(set(found))

In [126]:
# Re-run Extraction

df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)

In [127]:
# Check

(df["skill_count"] == 0).sum()

0

In [128]:
# Quick Role vs. Skill Quality Check 

df[["role", "title", "skills", "skill_count"]].sample(10, random_state=42)

,role,title,skills,skill_count
609,data engineer,Projektportfolio Agile Flow Lead (w/m/d),[r],1
539,data engineer,Electrical Engineer in Product Care 80 – 100 %...,"[git, r]",2
693,data engineer,Ingenieur Maschinenbau / Verfahrenstechnik - T...,[r],1
350,data engineer,Process Engineer Oats,"[excel, r]",2
174,data analyst,IT Business Analyst & Project Manager,"[git, excel, r, scala]",4
81,data scientist,Senior Scientist/Coordinator for high throughp...,"[excel, r]",2
355,data engineer,Data Ingestion Engineer (all),"[dashboard, azure, snowflake, data engineering...",11
424,data engineer,Principal/Staff Software Engineer,"[kubernetes, scala, kafka, git, spark, java, r...",12
523,data engineer,ICT System Engineer – Datacenter & Workplace 8...,"[git, excel, azure, r]",4
617,data engineer,Senior Azure Spezialist,"[azure, git, r, terraform, ci/cd, gitlab]",6


In [129]:
# Check Skill Distribution

df["skill_count"].describe()

count    743.000000
mean       4.197847
std        3.560777
min        1.000000
25%        2.000000
50%        3.000000
75%        6.000000
max       26.000000
Name: skill_count, dtype: float64

In [130]:
# Find Suspicious Rows

df[df["skill_count"] > 15][["title","skills"]].head()

,title,skills
30,Full-Stack Data Scientist,"[dashboard, plotly, data visualization, nlp, s..."
42,AI Engineer,"[snowflake, nlp, data engineering, r, sql, ci/..."
122,Data Analytics Engineer (H/F/X),"[dashboard, data engineering, r, sql, ci/cd, e..."
129,Data Scientist,"[r, sql, ci/cd, tensorflow, xgboost, databrick..."
236,AI Implementation Engineer,"[dashboard, nlp, r, ci/cd, tensorflow, gcp, fo..."


In [131]:
# Check Most Common Skills

from collections import Counter

all_skills = [skill for skills in df["skills"] for skill in skills]

Counter(all_skills).most_common(20)

[('r', 743),
 ('excel', 310),
 ('git', 276),
 ('python', 165),
 ('scala', 122),
 ('sql', 116),
 ('azure', 96),
 ('ci/cd', 88),
 ('machine learning', 86),
 ('aws', 72),
 ('kubernetes', 55),
 ('data analysis', 50),
 ('docker', 49),
 ('java', 46),
 ('linux', 45),
 ('dashboard', 40),
 ('data engineering', 37),
 ('c++', 37),
 ('terraform', 36),
 ('etl', 35)]

In [132]:
# The Fix for "r" appearing in 743 jobs 

import re

def extract_skills(row):
    text = " ".join([
        str(row["title"]),
        str(row["description"]),
        str(row["role"])
    ]).lower()

    found = []

    for skill in skill_patterns:
        skill_lower = skill.lower()

        # special case for R
        if skill_lower == "r":
            if re.search(r"(?<![a-z0-9])r(?![a-z0-9])", text):
                found.append(skill)

        # special case for c++
        elif skill_lower == "c++":
            if re.search(r"\bc\+\+\b", text):
                found.append(skill)

        # special case for ci/cd
        elif skill_lower == "ci/cd":
            if re.search(r"\bci/cd\b", text):
                found.append(skill)

        # normal case for all other skills
        else:
            if skill_lower in text:
                found.append(skill)

    return sorted(list(set(found)))

In [133]:
# Re-run skill extraction
df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)

In [134]:
# Check skill distribution again
df["skill_count"].describe()

count    743.000000
mean       3.596231
std        3.479410
min        0.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       26.000000
Name: skill_count, dtype: float64

In [135]:
# Check how many jobs still have 0 skills
(df["skill_count"] == 0).sum()

51

In [136]:
# Check how often R appears now
sum("r" in skills for skills in df["skills"])

331

In [137]:
# Last Check

df[["role","title","skills"]].sample(10, random_state=42)

,role,title,skills
609,data engineer,Projektportfolio Agile Flow Lead (w/m/d),[r]
539,data engineer,Electrical Engineer in Product Care 80 – 100 %...,[git]
693,data engineer,Ingenieur Maschinenbau / Verfahrenstechnik - T...,[r]
350,data engineer,Process Engineer Oats,"[excel, r]"
174,data analyst,IT Business Analyst & Project Manager,"[excel, git, scala]"
81,data scientist,Senior Scientist/Coordinator for high throughp...,[excel]
355,data engineer,Data Ingestion Engineer (all),"[azure, ci/cd, dashboard, data engineering, ex..."
424,data engineer,Principal/Staff Software Engineer,"[aws, ci/cd, flink, git, java, kafka, kubernet..."
523,data engineer,ICT System Engineer – Datacenter & Workplace 8...,"[azure, excel, git]"
617,data engineer,Senior Azure Spezialist,"[azure, ci/cd, git, gitlab, r, terraform]"


### Step 7: Salary Transparency Reconstruction

In this step, salary transparency is reconstructed from the raw salary field contained in the scraped dataset.

A binary variable, salary_available, is created to indicate whether a job posting explicitly provides salary information. The variable is coded as follows:

1 = salary information is available

0 = no salary information is provided

This transformation simplifies the raw salary field into a format that can later be used for descriptive analysis and comparisons across roles, seniority levels, or regions.

#### Observation

The `salary_available` variable captures whether salary information is disclosed in the job posting and serves as the basis for the later analysis of salary transparency. Since many job boards do not require salary disclosure, this variable is particularly useful for identifying transparency patterns across different segments of the job market.

In [138]:
# Create binary indicator for salary transparency
df["salary_available"] = df["salary_raw"].notna().astype(int)

# Quick check
df["salary_available"].value_counts(dropna=False)

salary_available
0    743
Name: count, dtype: int64

### Observation: Salary Transparency

The reconstruction of the salary transparency indicator shows that none of the 743 job postings include explicit salary information. This suggests that salary disclosure is very limited in the analyzed job postings. Such behavior is common on many job boards where salary disclosure is optional rather than mandatory.

As a result, the variable `salary_available` does not exhibit variation in the current dataset and therefore cannot be used for statistical comparisons within this sample. Nevertheless, the variable is retained in the dataset to maintain consistency with the overall data schema and to allow future integration with datasets that may contain salary information.

### Step 8: Seniority Classification

In this step, job seniority is inferred from the text of the job title and job description using a rule-based keyword matching approach.

The following seniority categories are used:

- intern

- junior

- mid

- senior

- lead

The classification is based on common seniority-related keywords such as intern, junior, senior, lead, or principal. If no explicit indicator is found, the posting is classified as mid by default.

This variable is useful for later analysis because it allows job postings to be grouped by experience level and makes it possible to compare skill requirements or salary transparency across different seniority categories.

#### Observation

The classification provides a simplified but practical approximation of experience levels across job postings. Because job advertisements do not always state seniority explicitly, a rule-based approach offers a transparent and reproducible way to derive this variable. Some classification noise may remain, but the resulting categories are sufficient for exploratory analysis of the structure of the data-related job market.

In [139]:
def classify_seniority(row):

    title = str(row["title"]).lower()
    description = str(row["description"]).lower()

    # title-based rules (most reliable)
    if any(x in title for x in ["intern", "internship", "trainee", "working student", "stagiaire"]):
        return "intern"

    elif any(x in title for x in ["junior", "entry level", "graduate"]):
        return "junior"

    elif any(x in title for x in ["senior", "sr."]):
        return "senior"

    elif any(x in title for x in ["lead", "principal", "head", "staff"]):
        return "lead"

    # fallback if no indicator found
    else:
        return "mid"

In [143]:
df["seniority"] = df.apply(classify_seniority, axis=1)
df["seniority"].value_counts(dropna=False)

seniority
mid       530
senior    125
lead       42
intern     26
junior     20
Name: count, dtype: int64

### Observation: Seniority Distribution

The seniority classificatio focuses primarily on job titles rather than full descriptions, to ensure the resulting distribution of seniority levels appears more realistic.

The majority of postings (approximately 70%) are classified as **mid-level roles**, which is typical for technology and data-related job markets where employers often seek candidates with some prior professional experience.

Senior roles account for a meaningful portion of the dataset, indicating demand for experienced professionals. Lead roles represent a smaller share, reflecting the limited number of managerial or highly specialized senior positions.

Entry-level positions, including internships and junior roles, appear less frequently, which is consistent with typical hiring patterns in technical fields where companies prioritize candidates with practical experience.

In [142]:
# Validation of newly created variables

print("Salary availability:")
print(df["salary_available"].value_counts(dropna=False))
print()

print("Seniority distribution:")
print(df["seniority"].value_counts(dropna=False))
print()

Salary availability:
salary_available
0    743
Name: count, dtype: int64

Seniority distribution:
seniority
NaN    743
Name: count, dtype: int64



### Step 10: Freeze Point - City-to-Canton Mapping

The preprocessing pipeline intentionally stops before the city-to-canton mapping step.

City-to-canton mapping will be implemented and refined separately in order to allow focused development and debugging of the location mapping logic.

This ensures the main preprocessing pipeline remains stable and reproducible while the location mapping step can be iteratively improved.

In [141]:
# --- LOCATION CLEANING ---
# clean_city()
# city_to_canton
# canton_to_region

In [144]:
# Save frozen preprocessing state before city mapping

df.to_csv("jobs_micro_preprocessed_freeze.csv", index=False)

print("Dataset saved at preprocessing freeze point.")
print("Shape:", df.shape)

Dataset saved at preprocessing freeze point.
Shape: (743, 21)
